# `01_boundaries`: Spatial reference units for the Netherlands (municipalities, provinces, H3 grid)

## Introduction

### Purpose

This notebook constructs the spatial reference units used throughout the pipeline. It loads the 2025 administrative boundaries of the Netherlands from PDOK, derives the municipality and province layers, generates an H3 hexagonal grid at resolution 8, and exposes the national boundary used by upstream notebooks for clipping. Per the thesis (§3.2, §3.4.6), the municipality is the unit of analysis for the headline classifiability results; provinces and the H3 grid feed the MAUP robustness check in stage 08.

### Inputs

- PDOK BRK 2025 GeoPackage (Bestuurlijke Gebieden, Atom distribution).

### Outputs

- `municipality_admin_areas_gdf` / `municipality_admin_areas_rel`: 342 Dutch municipalities (2025).
- `province_admin_areas_gdf` / `province_admin_areas_rel`: 12 Dutch provinces.
- `h3_cell_areas_gdf` / `h3_cell_areas_rel`: H3 hexagonal grid at resolution 8 (approximately 1 km² per cell).
- `nl_admin_area_gdf`: national land area boundary. Used by `00_bicycle_route_relations` and `00_non_bicycle_route_ways` to clip OSM ways to the Netherlands.

### Key steps

Each spatial level (municipality, province, country boundary) is read from the same PDOK 2025 GeoPackage. Geometries are reprojected from Amersfoort / RD New (EPSG:28992) to WGS84 (EPSG:4326) for compatibility with OSM data, and areas are computed on a separate reprojection to ETRS89 / LAEA Europe (EPSG:3035), an equal-area projection appropriate for square-kilometre calculations. The H3 grid is generated from the national boundary using the DuckDB H3 community extension; cell areas come from the extension's `h3_cell_area` function rather than from a reprojection.

### Dependencies on prior notebooks

None. This notebook is upstream of all other notebooks in the pipeline.

### Table of contents

1. [Environment setup](#1-environment-setup)
2. [Municipality](#2-municipality)
3. [Province](#3-province)
4. [H3 grid](#4-h3-grid)

---

---

## 1. Environment setup

### Libraries and extensions

In [1]:
import geopandas as gpd
import duckdb
from lonboard import viz
import pandas as pd
import requests

### Install DuckDB extensions

Load the spatial extension (vector geometry support) and the H3 community extension (hexagonal grid generation and cell-area computation).

In [2]:
duckdb.sql("INSTALL spatial; LOAD spatial;")

In [3]:
duckdb.sql("INSTALL h3 FROM community; LOAD h3;")

---

## 2. Municipality

As of 2025, the Netherlands consists of 342 municipalities. Spatial data describing these administrative areas is obtained from PDOK (Publieke Dienstverlening Op de Kaart), the central Dutch government portal for open geospatial data. The boundaries themselves derive from the Basic Registration of the Cadastre (BRK), the authoritative cadastral source for land parcels and administrative divisions in the Netherlands.

PDOK provides an [OGC API Features service](https://api.pdok.nl/kadaster/brk-bestuurlijke-gebieden/ogc/v1) for accessing administrative areas, but that endpoint only exposes the most recent version of the dataset and does not support historical snapshots. Since this analysis requires the 2025 snapshot specifically, the data is retrieved via the [PDOK Atom download service](https://www.pdok.nl/introductie/-/article/bestuurlijke-gebieden), which publishes versioned GeoPackage files for past annual releases. The 2025 dataset is documented in the [Nationaal Georegister](https://www.nationaalgeoregister.nl/geonetwork/srv/dut/catalog.search#/metadata/04d7ce1c-32dc-42d0-89a9-a642d6bc5e45).

The source data uses the Dutch national CRS Amersfoort / RD New (EPSG:28992). Geometries are reprojected to WGS84 (EPSG:4326) for downstream compatibility with OSM data; areas are computed separately in EPSG:3035 (an equal-area projection) and reported in km². The identifier and name columns are renamed to `municipality_code` and `municipality_name`. The results are stored as a GeoDataFrame (`municipality_admin_areas_gdf`) and a DuckDB-compatible Arrow table (`municipality_admin_areas_rel`).

In [4]:
# Download Dutch administrative areas (2025 snapshot) from the PDOK Atom distribution service.
# This GeoPackage contains multiple layers (municipalities, provinces, national boundary, etc.).
url = "https://service.pdok.nl/kadaster/brk-bestuurlijke-gebieden/atom/downloads/BestuurlijkeGebieden_2025.gpkg"

# Read the municipality layer ("gemeentegebied") from the GeoPackage.
municipality_admin_areas_gdf = gpd.read_file(url, layer='gemeentegebied')

# Data is provided in the Dutch national coordinate reference system (EPSG:28992 - RD New).
# Reproject geometry column to WGS84 ('EPSG:4326')
municipality_admin_areas_gdf.to_crs('EPSG:4326', inplace=True)

# Keep only the municipality identifier, municipality name, and geometry columns needed for further analysis.
municipality_admin_areas_gdf = municipality_admin_areas_gdf[['identificatie', 'naam', 'geometry']]

# Rename columns to more descriptive names.
municipality_admin_areas_gdf = municipality_admin_areas_gdf.rename(columns={'identificatie' : 'municipality_code',
                                     'naam' : 'municipality_name'})

# Calculate municipality area in square kilometers.
# First reproject geometries to EPSG:3035 (European equal-area projection)
# to ensure area calculations are accurate, then convert m² to km².
municipality_admin_areas_gdf['area_km2'] = municipality_admin_areas_gdf['geometry'].to_crs('EPSG:3035').area / 1_000_000

# Convert province_admin_areas into DuckDB relation
municipality_admin_areas_rel = municipality_admin_areas_gdf.to_arrow()

In [5]:
# Results are stored in two formats:
municipality_admin_areas_rel
municipality_admin_areas_gdf

,municipality_code,municipality_name,geometry,area_km2
0,GM0263,Maasdriel,"MULTIPOLYGON (((5.26613 51.7393, 5.26704 51.73...",75.473785
1,GM0441,Schagen,"MULTIPOLYGON (((4.71723 52.70422, 4.7174 52.70...",187.298445
2,GM1903,Eijsden-Margraten,"MULTIPOLYGON (((5.69601 50.75503, 5.69661 50.7...",78.759502
3,GM0193,Zwolle,"MULTIPOLYGON (((6.10191 52.46469, 6.10193 52.4...",119.379374
4,GM1711,Echt-Susteren,"MULTIPOLYGON (((5.85719 51.02854, 5.85921 51.0...",104.617927
...,...,...,...,...
337,GM0327,Leusden,"MULTIPOLYGON (((5.38028 52.09033, 5.38094 52.0...",58.903563
338,GM0553,Lisse,"MULTIPOLYGON (((4.55369 52.22225, 4.55393 52.2...",16.056337
339,GM0080,Leeuwarden,"MULTIPOLYGON (((5.77725 53.04798, 5.7788 53.04...",255.058466
340,GM0772,Eindhoven,"MULTIPOLYGON (((5.50524 51.40348, 5.50569 51.4...",89.371793


## 3. Province

As of 2025, the Netherlands consists of 12 provinces. The provincial boundaries are extracted from the `provinciegebied` layer of the same PDOK GeoPackage used for municipalities and undergo the same preprocessing: reprojection to EPSG:4326, renaming of the identifier and name columns to `province_code` and `province_name`, and area computation in km² via EPSG:3035.

The results are stored as a GeoDataFrame (`province_admin_areas_gdf`) and a DuckDB-compatible Arrow table (`province_admin_areas_rel`). Provinces enter the pipeline only as one of the alternative aggregation units in the MAUP robustness check (stage 08), not as the headline unit of analysis.

In [6]:
# Read the province layer ("provinciegebied") from the GeoPackage.
province_admin_areas_gdf = gpd.read_file(url, layer='provinciegebied')

# Data is provided in the Dutch national coordinate reference system (EPSG:28992 - RD New).
# Reproject geometry column to WGS84 ('EPSG:4326')
province_admin_areas_gdf.to_crs('EPSG:4326', inplace=True)

# Keep only the municipality identifier, municipality name, and geometry columns needed for further analysis.
province_admin_areas_gdf = province_admin_areas_gdf[['identificatie', 'naam', 'geometry']]

# Rename columns to more descriptive names.
province_admin_areas_gdf = province_admin_areas_gdf.rename(columns={'identificatie' : 'province_code',
                                     'naam' : 'province_name'})

# Calculate municipality area in square kilometers.
# First reproject geometries to EPSG:3035 (European equal-area projection)
# to ensure area calculations are accurate, then convert m² to km².
province_admin_areas_gdf['area_km2'] = province_admin_areas_gdf['geometry'].to_crs('EPSG:3035').area / 1_000_000

# Convert province_admin_areas into DuckDB relation
province_admin_areas_rel = province_admin_areas_gdf.to_arrow()

In [7]:
# Results are stored in two formats:
province_admin_areas_rel
province_admin_areas_gdf

,province_code,province_name,geometry,area_km2
0,PV22,Drenthe,"MULTIPOLYGON (((6.51623 52.63015, 6.51614 52.6...",2680.423696
1,PV24,Flevoland,"MULTIPOLYGON (((5.42666 52.25332, 5.45175 52.2...",2412.665944
2,PV21,Fryslân,"MULTIPOLYGON (((5.89872 52.80864, 5.89877 52.8...",5753.318073
3,PV25,Gelderland,"MULTIPOLYGON (((5.76897 51.75238, 5.7693 51.75...",5137.052289
4,PV20,Groningen,"MULTIPOLYGON (((7.02685 52.91904, 7.027 52.919...",2954.841372
5,PV31,Limburg,"MULTIPOLYGON (((6.0186 50.76368, 6.01872 50.76...",2209.879666
6,PV30,Noord-Brabant,"MULTIPOLYGON (((5.5706 51.22174, 5.57167 51.22...",5082.601383
7,PV27,Noord-Holland,"MULTIPOLYGON (((5.04666 52.16598, 5.04668 52.1...",4092.396642
8,PV23,Overijssel,"MULTIPOLYGON (((6.74938 52.11862, 6.74954 52.1...",3421.058822
9,PV26,Utrecht,"MULTIPOLYGON (((5.05181 51.85748, 5.05214 51.8...",1560.321591


## 4. H3 grid

H3 hexagonal cells are generated using the DuckDB H3 extension at resolution 8 (approximately 1 km² per cell), providing a balance between spatial detail and computational efficiency for national-scale analysis. The national boundary of the Netherlands is read from the `landgebied` layer of the same PDOK GeoPackage and serves as the spatial base for the grid.

As H3 requires geographic coordinates, the boundary is reprojected to EPSG:4326. Because H3 only accepts simple polygons, the MultiPolygon country geometry is first exploded into individual polygons (`ST_Dump`); each polygon is then approximated by a set of resolution-8 hexagons via `h3_polygon_wkt_to_cells`. Cell areas are computed in km² using the extension's `h3_cell_area` function (no separate reprojection is required, since the H3 area function returns geodesic areas directly).

The results are stored as a GeoDataFrame (`h3_cell_areas_gdf`) and a DuckDB-compatible Arrow table (`h3_cell_areas_rel`). Like provinces, the H3 grid enters the pipeline only via the MAUP robustness check in stage 08.

This cell also assigns `nl_admin_area_gdf`, the national land area boundary used by `00_bicycle_route_relations` and `00_non_bicycle_route_ways` to clip OSM ways to the Netherlands. That side-output is what allows downstream notebooks to import all required boundary variables through a single `%run` of this notebook.

In [8]:
# Load national boundary (Netherlands land area) from PDOK GeoPackage
# Layer "landgebied" contains the country area
nl_admin_area_gdf = gpd.read_file(url, layer='landgebied')

# Convert from Dutch national CRS (EPSG:28992 - RD New) to WGS84 (EPSG:4326)
# H3 requires latitude/longitude coordinates, not projected CRS
nl_admin_area_gdf.to_crs('EPSG:4326', inplace=True)

# Convert GeoDataFrame to Apache Arrow for efficient transfer to DuckDB without manual serialization
# Passing a GeoDataFrame directly into duckdb.sql() is not possible
nl_admin_area_arrow = nl_admin_area_gdf.to_arrow()

h3_cell_areas_rel = duckdb.sql("""
WITH explode_multipolygon AS (
    -- H3 requires simple polygon inputs
    -- If geometry is a MultiPolygon, split it into individual Polygon features
    SELECT * REPLACE(unnest(ST_Dump(geometry)).geom AS geometry)
    FROM nl_admin_area_arrow
),
h3_index AS (
    -- Convert polygon geometries into H3 cells at resolution 8
    -- Each polygon is approximated by a set of H3 hexagons
    SELECT *,
        unnest(h3_polygon_wkt_to_cells(ST_AsText(geometry), 8)) AS h3_index
    FROM explode_multipolygon
)
-- Final output:
-- 1. H3 index 
-- 2. Attach H3 geometry for each index
-- 3. Compute area of each H3 cell in square kilometers
SELECT 
    h3_index,
    h3_cell_to_boundary_wkt(h3_index)::geometry AS geometry,
    h3_cell_area(h3_index, 'km^2') AS area_km2
FROM h3_index
""")

# Convert h3_cell_areas_rel into GeoDataFrame
h3_cell_areas_gdf = gpd.GeoDataFrame.from_arrow(h3_cell_areas_rel.arrow())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [9]:
# Results are stored in two formats:
h3_cell_areas_rel 
h3_cell_areas_gdf 

,h3_index,geometry,area_km2
0,612936790103293951,"POLYGON ((5.40548 53.40924, 5.40383 53.405, 5....",0.606097
1,613046224768991231,"POLYGON ((5.03422 51.5353, 5.03265 51.53095, 5...",0.629743
2,612936901329944575,"POLYGON ((6.1175 53.4035, 6.1158 53.39926, 6.1...",0.610194
3,612936580008509439,"POLYGON ((5.70276 51.72289, 5.70113 51.71856, ...",0.631315
4,612936888971427839,"POLYGON ((6.87114 52.96407, 6.86939 52.95981, ...",0.620623
...,...,...,...
66525,612936582885801983,"POLYGON ((5.62648 51.90823, 5.62486 51.9039, 5...",0.628322
66526,612936891848720383,"POLYGON ((6.79838 53.12068, 6.79663 53.11642, ...",0.617980
66527,612936755905036287,"POLYGON ((4.67478 51.9506, 4.67323 51.94628, 4...",0.621899
66528,612936867131686911,"POLYGON ((5.68548 53.06579, 5.68382 53.06153, ...",0.612564
